# Transpile Input Circuits → Graphix Patterns (JSONL export)

_Prepared for distribution (Python 3.11.14 • Graphix v0.3.3)_

This notebook reads per‑subset OpenQASM 3 circuits from a second directory, transpiles them to Graphix patterns, optionally validates each pattern against the originating circuit, and exports newline‑delimited JSON (**.jsonl**) with metadata. 

**Layout:** `Imports → Configuration → Helpers → Main`

In [1]:

# %% [code]
# Imports
import sys
import json
import re
import ast
from numpy import pi
from pathlib import Path
from typing import Optional, Dict, Any
import numpy as np

# graphix imports
from graphix import Circuit, pattern
from graphix.parameter import Placeholder
from graphix.states import Plane
from graphix import gflow
import graphix as _graphix

# local helpers (shipped alongside this notebook)
from qasm_to_graphix import build_graphix_from_qasm, circuit_to_compact_string
from bind_pattern_params import bind_pattern_by_names

# parser version used by qasm importer
import openqasm3 as _openqasm3

# execution helper
from runpy import run_path


In [2]:
# %% [code]
# Configuration (edit here)
NAME_TO_FIND = "Be2"  
#All Pauli strings up to 5 qubits: arbitrary1 (1),arbitrary2 (2), arbitrary3 (3), arbitrary4 (4), arbitrary5 (5)
#Molecules: Be2, H2 (8), BH (10), OH (12), NH (14), Li2 (14), C2 (18), N2 (22), Na2 (24)
COLORING_STRATEGY = "one_to_one" # "one_to_one" or "smallest_last" 
HAM_INSTANCE_TO_FIND = "ham_JW"   # molecules: 'ham_JW'; tfim: specific graph instance id; arbitrary: "arbitrary"
#For tfim choose one of: 
#   graph-1D-grid-pbc-qubitnodes_Lx-16_h-2, graph-1D-grid-pbc-qubitnodes_Lx-26_h-6,
#   graph-2D-grid-nonpbc-qubitnodes_Lx-5_Ly-15_h-3, graph-2D-triag-pbc-qubitnodes_Lx-13_Ly-13_h-0.1,
#   graph-2D-triag-pbc-qubitnodes_Lx-19_Ly-19_h-0.5, graph-2D-triag-nonpbc-qubitnodes_Lx-3_Ly-160_h-0.1,
#   graph-2D-grid-nonpbc-qubitnodes_Lx-19_Ly-19_h-5, graph-2D-grid-pbc-qubitnodes_Lx-4_Ly-148_h-6

# I/O locations: cicuit files should be generated by the first pipeline (evolution_to_gates) and placed in: INPUT_CIRCUIT_DIR. 
# Output patterns will be placed in the json_output folder: OUTPUT_JSON_DIR.
INPUT_CIRCUIT_DIR = Path(f"../Circuit_Code/qasm_circuit_files_{NAME_TO_FIND}_{COLORING_STRATEGY}/")
INPUT_CIRCUIT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_JSON_DIR = Path(f"json_output_pattern_{NAME_TO_FIND}_{COLORING_STRATEGY}/")
OUTPUT_JSON_DIR.mkdir(parents=True, exist_ok=True)

# Options
PRINT_PATTERNS = False # set to True to print the full pattern for each circuit
DRAW_GRAPHS_FROM_PATTERN = False # set to True to draw graphs structures from the generated patterns (not saved)
PERFORM_CIRCUIT_TO_PATTERN_TEST = True # True only for low qubit counts, as it requires simulating the full circuit and pattern
FIDELITY_TEST_THRESHOLD = 0.999 # set to a value between 0 and 1 to determine whether patterns pass the fidelity test 
EXPORT_FULL_PATTERN_METADATA_STATS = True # set to True to export metadata for full patterns
TEST_PATTERNS_AGAINST_FULL_HAMILTONIAN = True # True only for low qubit counts
COMPUTE_FULL_OVERLAP = True # True only for 6 qubits or fewer, Be2
NUMBER_AMPLITUDES_TO_CHECK = 5 # number of amplitudes to check for correctness when testing patterns 
RANDOM_SEED = 1234 # set random seed for reproducibility of pattern generation and testing
SIM_BACKEND = "tensornetwork" # "statevector" (exact, but memory intensive), "tensornetwork" (approximate, but can handle more qubits), "stabilizer" (exact for Clifford circuits, very fast and low memory)
MAX_RECURSION_DEPTH = 500000 #increase recursion limit for large patterns

# Derived / misc
FULL_CIRCUIT_DIR = Path(f"../Circuit_Code/full_circuit_files_{NAME_TO_FIND}/")
FULL_CIRCUIT_DIR.mkdir(parents=True, exist_ok=True)

COUPLING = "none"                 # current transpilation: 'none' (future: e.g., 'linear')
# Coupling "none" is the default, and corresponds to no relabeling of the qubits 
# Other coupling strategies (e.g., "linear") are not supported in this notebook and will require relabeling of 
# the qubits in the transpiled circuits, which is not implemented here.
if COUPLING != "none":
    raise NotImplementedError(f"Coupling strategy '{COUPLING}' is not supported in this notebook. Please set COUPLING to 'none'.")


# Metadata path (generated by the first pipeline)
INPUT_META_FILE = INPUT_CIRCUIT_DIR / (
    f"meta_data_evolution_to_gates_for_graphix_{NAME_TO_FIND}_{HAM_INSTANCE_TO_FIND}_map_{COUPLING}_{COLORING_STRATEGY}.py"
)




In [3]:
# %% [code]
# Helpers

def build_random_rotations_circuit(
    nqubits: int,
    *,
    scheme: str = "bloch",              # "bloch" (random state) or "haar" (random unitary)
    rng: Optional[np.random.Generator] = None
) -> Circuit:
    """
    Build a Graphix circuit on `nqubits` (default input |+>^n),
    then apply a random single-qubit rotation to each qubit using only {H, RZ}.
    """
    if nqubits < 1:
        raise ValueError("nqubits must be >= 1.")
    rng = rng or np.random.default_rng()

    circ = Circuit(nqubits)  # Graphix circuits default to |+> on each wire.

    if scheme.lower() == "bloch":
        for q in range(nqubits):
            u, v = rng.random(), rng.random()
            theta = np.arccos(1.0 - 2.0*u)
            phi   = 2.0 * pi * v
            circ.h(q)
            circ.rz(q, phi)
            # Ry(theta) via {H, RZ}
            circ.rz(q,  pi/2); circ.h(q); circ.rz(q,  theta); circ.h(q); circ.rz(q, -pi/2)
    elif scheme.lower() == "haar":
        for q in range(nqubits):
            alpha = 2.0*pi*rng.random()
            gamma = 2.0*pi*rng.random()
            u     = rng.random()
            beta  = np.arccos(1.0 - 2.0*u)
            circ.rz(q, alpha)
            circ.h(q); circ.rz(q, beta); circ.h(q)  # Rx(beta)
            circ.rz(q, gamma)
    else:
        raise ValueError("scheme must be 'bloch' or 'haar'.")
    return circ

def compose_two(p_left, p_right):
    mapping = {i2: o1 for i2, o1 in zip(p_right.input_nodes, p_left.output_nodes)}
    p_lr, _ = p_left.compose(p_right, mapping, preserve_mapping=True)
    return p_lr

def sanitize_pattern_ascii(ascii_text: str, drop_layers: bool = False) -> str:
    s = ascii_text
    s = re.sub(r'(?<![\d.])-0(?:\.0+)?', '0', s)  # -0 / -0.0 -> 0 / 0.0
    s = re.sub(r'\s*\+\s*\(\s*0(?:\.0+)?\s*\)', '', s)
    s = re.sub(r'\s*-\s*\(\s*0(?:\.0+)?\s*\)', '', s)
    s = re.sub(r'\s*\+\s*0(?:\.0+)?', '', s)
    s = re.sub(r'\s*-\s*0(?:\.0+)?', '', s)
    s = re.sub(r'\(\s*0(?:\.0+)?\s*\)', '(0)', s)
    if drop_layers:
        s = re.sub(r'\{\d+\}', '', s)
    s = re.sub(r'[ ]{2,}', ' ', s).strip()
    return s

def get_causal_flow_layers_lean(pat):
    graph = pat.extract_graph()
    v_in = set(pat.input_nodes or [])
    v_out = set(pat.output_nodes or [])
    meas_planes = pat.get_meas_plane()
    meas_angles = pat.get_angles()

    measured_nodes = set(graph.nodes()) - v_out
    bad = {n: meas_planes.get(n, None)
           for n in measured_nodes
           if meas_planes.get(n, Plane.XY) != Plane.XY}
    if bad:
        raise ValueError("gflow.find_flow requires Plane.XY; non-XY planes found.")

    paulix_list, pauliy_list, _ = gflow.get_pauli_nodes(meas_planes, meas_angles)
    meas_planes_xy = {n: Plane.XY for n in graph.nodes()}
    node_list, layer_map = gflow.find_flow(graph, v_in, v_out, meas_planes=meas_planes_xy)
    return node_list, layer_map, paulix_list, pauliy_list

def parse_metadata_py_to_json(input_path: str, output_path: Optional[str] = None) -> Dict[str, Any]:
    """Parse metadata .py file (from the first pipeline) to a dict.
    Only returns the dict; if *output_path* is provided, writes JSON there.
    """
    text = Path(input_path).read_text(encoding='utf-8')
    meta_comment = None
    m0 = re.search(r"^\s*#\s*Metadata.*$", text, re.MULTILINE)
    if m0:
        meta_comment = m0.group(0).strip('# ').strip()

    def extract_list(name: str) -> Any:
        m = re.search(rf"{name}\s*=\s*(\[[\s\S]*?\])", text)
        return ast.literal_eval(m.group(1)) if m else None

    m_nq = re.search(r"Number_qubits\s*=\s*(\d+)", text)
    number_qubits = int(m_nq.group(1)) if m_nq else None
    coefficients = extract_list('Coefficients')
    terms = extract_list('Terms')

    paulis_subsets: Dict[str, Any] = {}
    for m in re.finditer(r"commuting_paulis_(\d+)\s*=\s*(\[[\s\S]*?\])", text):
        idx = m.group(1)
        lst = ast.literal_eval(m.group(2))
        paulis_subsets[idx] = lst

    result: Dict[str, Any] = {
        "Hamiltonian": NAME_TO_FIND,
        "Instance": HAM_INSTANCE_TO_FIND,
        "Number_qubits": number_qubits,
        "Coefficients": coefficients,
        "Terms": terms,
        "coloring_strategy": COLORING_STRATEGY,
        "subset_index vs commuting_paulis": paulis_subsets,
    }
    if meta_comment:
        result["metadata_comment"] = meta_comment
    if output_path:
        Path(output_path).write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    return result


def max_edge_layer_span_from_graph_and_layers(graph, layer_map) -> int:
    """Return the maximum |layer[u] - layer[v]| over edges of `graph`,
    using a precomputed `layer_map` (e.g., from get_causal_flow_layers_lean).
    """
    max_span = 0
    # Stream over edges once; no list allocation.
    for u, v in graph.edges():
        du = layer_map[int(u)]
        dv = layer_map[int(v)]
        d = du - dv
        if d < 0:
            d = -d
        if d > max_span:
            max_span = d
    return int(max_span)

def export_molecule_patterns_jsonl(name_to_find: str, dir_path, patterns, fullpattern, indices=None, out_dir=OUTPUT_JSON_DIR):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"pattern_{name_to_find}_{HAM_INSTANCE_TO_FIND}_map_{COUPLING}_{COLORING_STRATEGY}.jsonl"

    # get metadata from .py file
    result = parse_metadata_py_to_json(
        dir_path / f"meta_data_evolution_to_gates_for_graphix_{name_to_find}_{HAM_INSTANCE_TO_FIND}_map_{COUPLING}_{COLORING_STRATEGY}.py"
    )
    from importlib.metadata import version, PackageNotFoundError
    try:
        _graphix.__version__ = version('graphix')
    except PackageNotFoundError:
        _graphix.__version__ = 'none'

    # --- add provenance ---
    result["provenance"] = {
        "python_version": __import__('platform').python_version(),
        "graphix_version": getattr(_graphix, '__version__', 'none'),
        "openqasm3_version": getattr(_openqasm3, '__version__', 'unknown'),
        "numpy_version": np.__version__,
        "sim_backend": SIM_BACKEND,
        "random_seed": RANDOM_SEED,
        "flags": {
            "PERFORM_CIRCUIT_TO_PATTERN_TEST": PERFORM_CIRCUIT_TO_PATTERN_TEST,
            "FIDELITY_TEST_THRESHOLD": FIDELITY_TEST_THRESHOLD,
            "EXPORT_FULL_PATTERN_METADATA_STATS": EXPORT_FULL_PATTERN_METADATA_STATS,
            "TEST_PATTERNS_AGAINST_FULL_HAMILTONIAN": TEST_PATTERNS_AGAINST_FULL_HAMILTONIAN,
            "COMPUTE_FULL_OVERLAP": COMPUTE_FULL_OVERLAP,
            "NUMBER_AMPLITUDES_TO_CHECK": NUMBER_AMPLITUDES_TO_CHECK
        }
    }

    if indices is None:
        indices = range(len(patterns)) if not isinstance(patterns, dict) else sorted(patterns.keys())

    # accumulate global stats
    global_depth = 0
    global_max_degree = 0
    for idx in indices:
        pat = patterns[idx]
        degree = pat.compute_max_degree()
        global_max_degree = max(global_max_degree, degree)
        _, layer_list_causal_flow, paulix_list, pauliy_list = get_causal_flow_layers_lean(pat)
        global_depth += max(layer_list_causal_flow.values()) + 1
        print(f" Pattern {idx}: global max degree so far={global_max_degree} global_depth so far={global_depth}")

    result["concatenated_global_depth"] = global_depth
    result["global_max_degree"] = global_max_degree

    out_path.write_text(json.dumps(result, ensure_ascii=False) + "\n", encoding="utf-8")

    with out_path.open("a", encoding="utf-8") as f:
        if EXPORT_FULL_PATTERN_METADATA_STATS and fullpattern is not None:
            pat = fullpattern
            graph = pat.extract_graph()
            _, layer_list_causal_flow, paulix_list, pauliy_list = get_causal_flow_layers_lean(pat)
            row = {
                "Strategy for comparison": "full_hamiltonian",
                "meta": {
                    "node number": pat.n_node,
                    "input_nodes": list(pat.input_nodes),
                    "output_nodes": list(pat.output_nodes),
                    "max degree": pat.compute_max_degree(),
                    "max edge layer span": max_edge_layer_span_from_graph_and_layers(graph, layer_list_causal_flow),
                    "depth of full pattern": max(layer_list_causal_flow.values()) + 1,
                    "number Pauli X measurements": len(paulix_list),
                    "number Pauli Y measurements": len(pauliy_list)
                },
            }
            f.write(json.dumps(row, ensure_ascii=False) + "\n"); 
        for idx in indices:
            pat = patterns[idx]
            graph = pat.extract_graph()
            ascii_text = pat.to_ascii(left_to_right=True, limit=999_999)
            ascii_text = sanitize_pattern_ascii(ascii_text)
            node_list_cf, layer_list_cf, paulix_list, pauliy_list = get_causal_flow_layers_lean(pat)
            row = {
                "Commuting subset": int(idx),
                "meta": {
                    "node number": pat.n_node,
                    "input_nodes": list(pat.input_nodes),
                    "output_nodes": list(pat.output_nodes),
                    "max degree": pat.compute_max_degree(),
                    "number Pauli X measurements": len(paulix_list),
                    "number Pauli Y measurements": len(pauliy_list),
                    "number layers (causal flow)": max(layer_list_cf.values()) + 1,
                    "max edge layer span": max_edge_layer_span_from_graph_and_layers(graph, layer_list_cf),
                    "node_layer_list_causal_flow": {n: layer_list_cf[n] for n in sorted(layer_list_cf)}
                },
                "pattern_ascii": ascii_text,
            }
            f.write(json.dumps(row, ensure_ascii=False) + "\n"); 
    return out_path




In [4]:
# %% [code]
# Main


sys.setrecursionlimit(MAX_RECURSION_DEPTH) 
globals_dict = run_path(str(Path(INPUT_META_FILE)))
nqubits = globals_dict["Number_qubits"]
numerical_coeff = globals_dict["Coefficients"]
num_amplitudes_full_sv = 1 << nqubits  # integer power-of-two size

# Count subset circuits
num_patterns = sum(
    1 for p in INPUT_CIRCUIT_DIR.rglob(f"evolution*{NAME_TO_FIND}*{HAM_INSTANCE_TO_FIND}*{COUPLING}*{COLORING_STRATEGY}*")
    if p.is_file() and p.suffix.lower() == ".qasm" and NAME_TO_FIND.lower() in p.name.lower()
)
print("n_files", num_patterns)
if num_patterns == 0:
    print("Error: no input files found"); sys.exit(1)

# Placeholder handles
c = [Placeholder(f"c[{i}]") for i in range(len(numerical_coeff))]
patterns = []

# Primary loop: read subcircuits, transpile to patterns
for pattern_index in range(num_patterns):
    qasm_input_path = INPUT_CIRCUIT_DIR / (
        f"evolution_to_gates_for_graphix_{NAME_TO_FIND}_{HAM_INSTANCE_TO_FIND}_map_{COUPLING}_{COLORING_STRATEGY}_commsubgrp_{pattern_index}.qasm"
    )
    if not qasm_input_path.is_file():
        sys.exit(f"[ERROR] QASM file not found: {qasm_input_path.resolve()}")
    qasm_text = qasm_input_path.read_text(encoding="utf-8")

    circuit, params2 = build_graphix_from_qasm(qasm_text)
    p = circuit.transpile().pattern
    p.standardize(); p.shift_signals()
    patterns.append(p)

    if PRINT_PATTERNS:
        print(p.to_ascii(left_to_right=True, limit=10000))

    if PERFORM_CIRCUIT_TO_PATTERN_TEST:
        
        values = {f"c[{i}]": float(numerical_coeff[i]) for i in range(len(numerical_coeff))}
        ptest = bind_pattern_by_names(p, values, verify=True)
        ptest.standardize(); ptest.shift_signals()
        circ_initialstate = build_random_rotations_circuit(nqubits, scheme="bloch", rng=np.random.default_rng(RANDOM_SEED))
        initialstate_pattern = circ_initialstate.transpile().pattern
        initialstate_pattern.standardize(); initialstate_pattern.shift_signals()
        pattern_with_initial_state = compose_two(initialstate_pattern, ptest)
        pattern_with_initial_state.standardize(); pattern_with_initial_state.shift_signals()

        # Bind numeric values into the circuit copy and compare distributions
        ctest = circuit
        from qasm_to_graphix import circuit_to_compact_string

        for i, val in enumerate(numerical_coeff):
            name = f"c[{i}]"
            if name in params2:
                ctest = ctest.subs(params2[name], float(val))
        
        
        tn = pattern_with_initial_state.simulate_pattern(backend=SIM_BACKEND)
        p_pat=[0.0 for _ in range(num_amplitudes_full_sv)]
        for amp_indx in range(num_amplitudes_full_sv):
            p_pat[amp_indx] = tn.get_basis_amplitude(amp_indx)
        pattern_norm = sum(p_pat[amp_indx] for amp_indx in range(num_amplitudes_full_sv))
        
        input_state_circ = circ_initialstate.simulate_statevector().statevec.flatten()
        sv = ctest.simulate_statevector(input_state=input_state_circ).statevec.flatten()    
        
        circuit_norm = sum(float(np.abs(sv[amp_indx])**2) for amp_indx in range(len(sv)))

        if pattern_norm < FIDELITY_TEST_THRESHOLD or circuit_norm < FIDELITY_TEST_THRESHOLD:
            print(f"Warning: low norm detected. pattern norm: {pattern_norm} circuit norm: {circuit_norm}")
            sys.exit(1)
       
        fidelity_test = 0.0
        # Compare probability distributions via Bhattacharyya coefficient
        for amp_indx in range(num_amplitudes_full_sv):
            p_circ = float(np.abs(sv[amp_indx])**2)
            fidelity_test += np.sqrt(p_pat[amp_indx] * p_circ)
        
        if fidelity_test < FIDELITY_TEST_THRESHOLD:
            if pattern_index > 0:
                print(f" FAIL: fidelity of circuit vs. pattern {pattern_index}: {fidelity_test}")
                sys.exit(1)
        else:
            print(f" PASS pattern {pattern_index}: fidelity of circuit vs. pattern {fidelity_test}")

# Full Hamiltonian pattern (optional)
fullpattern = None
if EXPORT_FULL_PATTERN_METADATA_STATS or PRINT_PATTERNS or TEST_PATTERNS_AGAINST_FULL_HAMILTONIAN:
    qasm_path = FULL_CIRCUIT_DIR / (
        f"full_evolution_to_gates_for_graphix_{NAME_TO_FIND}_{HAM_INSTANCE_TO_FIND}_map_{COUPLING}_{COLORING_STRATEGY}.qasm"
    )
    if not qasm_path.is_file():
        sys.exit(f"[ERROR] QASM file not found: {qasm_path.resolve()}")
    qasm_text = qasm_path.read_text(encoding="utf-8")
    full_circuit, paramsfull = build_graphix_from_qasm(qasm_text)

if EXPORT_FULL_PATTERN_METADATA_STATS or TEST_PATTERNS_AGAINST_FULL_HAMILTONIAN:
    fullpattern = full_circuit.transpile().pattern
    fullpattern.standardize(); fullpattern.shift_signals()
    if PRINT_PATTERNS:
        print(fullpattern.to_ascii(left_to_right=True, limit=1000))

# Concatenation vs full Hamiltonian (optional)
if TEST_PATTERNS_AGAINST_FULL_HAMILTONIAN:
    if (NAME_TO_FIND not in ["Be2"]) and (not NAME_TO_FIND.startswith("arbitrary")):
    #if NAME_TO_FIND not in ["Be2"] or ["arbitrary"]:
        print("Warning: full patterns are memory intensive due to exponential scaling.")
    # Concatenate subsets; handle the single‑pattern case gracefully
    if len(patterns) == 1:
        p_all = patterns[0]
    else:
        p_all = compose_two(patterns[0], patterns[1])
        for pattern_index in range(2, len(patterns)):
            p_all = compose_two(p_all, patterns[pattern_index])
    p_all.standardize(); p_all.shift_signals()


    # Replace placeholders with numeric coefficients
    values = {f"c[{i}]": float(numerical_coeff[i]) for i in range(len(numerical_coeff))}
    p_all = bind_pattern_by_names(p_all, values, verify=True)
    fullpattern = bind_pattern_by_names(fullpattern, values, verify=True)

    print(" Simulating with tensornetwork backend ")
    ans_con = p_all.simulate_pattern(backend=SIM_BACKEND)
    ans0 = fullpattern.simulate_pattern(backend=SIM_BACKEND)
    for amp_indx in range(NUMBER_AMPLITUDES_TO_CHECK):
        amplitude_concatenated = ans_con.get_basis_amplitude(amp_indx)
        amplitude_full = ans0.get_basis_amplitude(amp_indx)
        print(f" amplitude {amp_indx} concatenated: ", amplitude_concatenated)
        print(f" amplitude {amp_indx} full: ", amplitude_full)

    if COMPUTE_FULL_OVERLAP:
        if (NAME_TO_FIND not in ["Be2"]) and (not NAME_TO_FIND.startswith("arbitrary")):
        #if NAME_TO_FIND not in ["Be2"] or ["arbitrary"]:
            print("Full overlap only feasible for small patterns."); sys.exit(1)
        print("computing overlap of concatenated and full states")
        sv = ans_con.to_statevector().flatten()
        av0 = ans0.to_statevector().flatten()
        print("overlap of states: ", float(np.abs(np.dot(av0.conjugate(), sv))))

# Export JSONL with metadata & stats
export_molecule_patterns_jsonl(NAME_TO_FIND, INPUT_CIRCUIT_DIR, patterns, fullpattern)

n_files 62
 PASS pattern 0: fidelity of circuit vs. pattern 0.9999999999999966
 PASS pattern 1: fidelity of circuit vs. pattern 0.9999999999999967
 PASS pattern 2: fidelity of circuit vs. pattern 0.9999999999999973
 PASS pattern 3: fidelity of circuit vs. pattern 0.9999999999999964
 PASS pattern 4: fidelity of circuit vs. pattern 0.9999999999999968
 PASS pattern 5: fidelity of circuit vs. pattern 0.9999999999999968
 PASS pattern 6: fidelity of circuit vs. pattern 0.9999999999999966
 PASS pattern 7: fidelity of circuit vs. pattern 0.9999999999999972
 PASS pattern 8: fidelity of circuit vs. pattern 0.999999999999997
 PASS pattern 9: fidelity of circuit vs. pattern 0.9999999999999968
 PASS pattern 10: fidelity of circuit vs. pattern 0.999999999999997
 PASS pattern 11: fidelity of circuit vs. pattern 0.9999999999999972
 PASS pattern 12: fidelity of circuit vs. pattern 0.9999999999999972
 PASS pattern 13: fidelity of circuit vs. pattern 0.9999999999999967
 PASS pattern 14: fidelity of circu

PosixPath('json_output_pattern_Be2_one_to_one/pattern_Be2_ham_JW_map_none_one_to_one.jsonl')

In [5]:


# Optional: draw graphs from patterns
if DRAW_GRAPHS_FROM_PATTERN:
    import importlib, draw_graphix_pattern_single
    importlib.reload(draw_graphix_pattern_single)

    from draw_graphix_pattern_single import draw_graphix_pattern
    
    # Optional: draw patterns
    # Choose pattern index to draw (e.g., 0 for the first pattern, or 31 for the last pattern in the case of Be2 with 32 patterns)
    INDEX_TO_DRAW = 5
    patterns[INDEX_TO_DRAW].draw_graph(flow_from_pattern=False,node_distance=(0.7, 0.6))
    fig, ax = draw_graphix_pattern(patterns[INDEX_TO_DRAW])
    print("\n Maximum degree of patterns[INDEX_TO_DRAW]:", patterns[INDEX_TO_DRAW].compute_max_degree())
    import copy
    pcompact = copy.deepcopy(patterns[INDEX_TO_DRAW])
    values = {f"c[{i}]": float(numerical_coeff[i]) for i in range(len(numerical_coeff))}
    pcompact = bind_pattern_by_names(pcompact, values, verify=True)
    pcompact.perform_pauli_measurements(leave_input=True)
    pcompact.standardize(); pcompact.shift_signals()
    pcompact.draw_graph(flow_from_pattern=True,node_distance=(0.7, 0.6),show_loop=False)
    fig, ax = draw_graphix_pattern(pcompact)
    print("\n Maximum degree of patterns[INDEX_TO_DRAW] after performing Pauli measurements:", pcompact.compute_max_degree())

